In [0]:
from pyspark.sql import functions as F

catalog = "fortum_challenge_data"
schema = "05_deliverables"
table = "consumption_deliverables_hourly"

df = spark.table(f"{catalog}.{schema}.{table}")

ordered_df = (
    df
    .orderBy("measured_at")
    .coalesce(1)
)

df_export = ordered_df.select(
    *[
        F.regexp_replace(F.col(c).cast("string"), "\\.", ",").alias(c)
        if dict(ordered_df.dtypes)[c] in ["double","float","decimal"]
        else F.col(c)
        for c in ordered_df.columns
    ]
)

df_export.write \
    .mode("overwrite") \
    .option("header", True) \
    .option("sep", ";") \
    .csv("/Volumes/fortum_challenge_data/05_deliverables/exports/consumption_deliverables_hourly_csv")